In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
plt.rcParams.update({
    "font.size" : 18,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})


In [ ]:
folderVec = [

    # "../data/fwh_diy_monopole_sphere_d20_f1s_gp3_m0/microphones/obs1",
    # "../data/fwh_diy_monopole_sphere_d20_f1s_gp3_m01/microphones/obs1"
    # "../data/fwh_diy_monopole_sphere_d20_f1s_gp3_m03/microphones/obs1",
    # "../data/fwh_diy_dipole_sphere_d20_f1s_gp3_m0/microphones/obs1"
    "../data/fwh_diy_dipole_sphere_d20_f1s_gp3_m03/microphones/obs1"
]

caseLblVec = [
    r'FW-H ($D_{sphere}=20$)  ',
]

#! for observers
obsLblVec = [
    r'$\theta = 0.0$',
    r'$\theta = \pi / 2$',
    # r'$\theta = \pi$',
    r'$\theta = (3/4)\pi$',
]

colorVec = [
    "#910000",
    "#119100",
    # "#001AFF",
    "#B700FF",
    # '#FF5733',
]

markerVec = [
    'o',
    '^',
    's',
    "P",
    '*',
]


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec
def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z


In [ ]:
# read observers (y)
obsVecVec = []
dataVecVec = []
obsTimeVecVec = []
for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/microphone*.txt"))
    coordVec = [[], [], []]
    dataVec = []
    obsTimeVec = []
    for filePath in fileList:
        x, y, z = get_probe_coords(filePath)
        coordVec[0].append(x)
        coordVec[1].append(y)
        coordVec[2].append(z)
        dataVec.append(read_col_probe(filePath, 2))
        obsTimeVec.append(read_col_probe(filePath, 1))
    obsVecVec.append(coordVec)
    dataVecVec.append(dataVec)
    obsTimeVecVec.append(obsTimeVec)



In [ ]:
dxPhy = 1.0
rho0Phy = 1.0
csPhy = 1.0 / np.sqrt(3)
gamma = 1.0
csLB = np.sqrt(gamma / 3.0)

CLength = dxPhy / 1.0
CRho = rho0Phy / 1.0
CVel = csPhy / csLB
CTime = CLength / CVel
CMass = CRho * CLength**3
CPressure = CMass / (CLength * CTime**2)

freqStep = 0.0017 # ~1s
# U0Phy = 0.0 #! M=0
# U0Phy = 0.057735027 #! M=0.1
U0Phy = 0.173205081 #! M-0.3

M0 = U0Phy / csPhy
ampLB = 1.0
ampPhy = ampLB * CRho
TPhy = (1.0 / freqStep) * CTime
omega = 2 * np.pi / TPhy


In [ ]:

#! reference (physical unit)
beta = np.sqrt(1.0 - M0**2)
betaSq = beta**2
pSrc = [0, 0, 0]
timeTotal = 10 * TPhy
dt = 0.001
tVec = np.linspace(0.0, timeTotal, int(round(timeTotal / dt)) + 1, True)

def monopole(pSrc, pObs, amp, omega, rho0, u0, cs, t):
    M0 = u0 / cs
    beta = np.sqrt(1.0 - M0**2)
    betaSq = beta**2
    RStar = np.sqrt((pObs[0] - pSrc[0])**2 + betaSq * ((pObs[1] - pSrc[1])**2 + (pObs[2] - pSrc[2])**2))
    R = (RStar - M0 * (pObs[0] - pSrc[0])) / betaSq
    RStarDerivX = (pObs[0] - pSrc[0]) / RStar
    RStarDerivY = betaSq * (pObs[1] - pSrc[1]) / RStar
    RStarDerivZ = betaSq * (pObs[2] - pSrc[2]) / RStar
    RDerivX = (RStarDerivX - M0) / betaSq
    RDerivY = RStarDerivY / betaSq
    RDerivZ = RStarDerivZ / betaSq

    factor = amp / (4 * np.pi * RStar) * np.exp(1j * omega * (t - R / cs))
    uXTmp = RStarDerivX / RStar + 1j * omega / cs * RDerivX
    uYTmp = RStarDerivY / RStar + 1j * omega / cs * RDerivY
    uZTmp = RStarDerivZ / RStar + 1j * omega / cs * RDerivZ
    uPrime = [-factor * uXTmp, -factor * uYTmp, -factor * uZTmp]
    pPrime = np.real(-rho0 * (1j * omega * factor + u0 * uPrime[0]))
    return pPrime

def dipole(pSrc, dy, pObs, amp, omega, rho0, u0, cs, t):
    pPrimeUpper = monopole([pSrc[0], pSrc[1] - dy, pSrc[2]], pObs, amp, omega, rho0, u0, cs, t)
    pPrimeLower = monopole([pSrc[0], pSrc[1] + dy, pSrc[2]], pObs, amp, omega, rho0, u0, cs, t)
    pPrime = (pPrimeUpper - pPrimeLower) / (2 * dy)
    return -pPrime

# uPrimeVecVec = []
pPrimeVecVec = []
RStarVecVec = []
for case in range(0, len(folderVec)):
    uPrimeVec = [[], [], []]
    pPrimeVec = []
    RStarVec = []
    for iObs in range(0, len(obsVecVec[case][0])):
        pObs = [obsVecVec[case][0][iObs], obsVecVec[case][1][iObs], obsVecVec[case][2][iObs]]
        # pPrime = monopole(pSrc, pObs, ampPhy, omega, rho0Phy, U0Phy, csPhy, tVec)
        pPrime = dipole(pSrc, 0.001, pObs, ampPhy, omega, rho0Phy, U0Phy, csPhy, tVec)
        pPrimeVec.append(pPrime)

    # uPrimeVecVec.append(uPrimeVec)
    pPrimeVecVec.append(pPrimeVec)
    RStarVecVec.append(RStarVec)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 5), facecolor='w', edgecolor='w')
for case in range(0, len(folderVec)):
    for iObs in range(0, len(obsVecVec[case][0])):
        ax.plot(tVec/TPhy, pPrimeVecVec[case][iObs], label=f"{obsLblVec[iObs]}  Analytical", lw=2, c=colorVec[iObs])
        ax.plot(obsTimeVecVec[case][iObs]/TPhy, dataVecVec[case][iObs], marker=markerVec[iObs], ms=10, markevery=30, fillstyle='none', lw=0, c=colorVec[iObs], label=f"{obsLblVec[iObs]} {caseLblVec[case]}")

ax.set_xlim(2, 6)
# ax.set_ylim(-1E-5, 1E-5)
# ax.set_ylim(-1.0E-7, 1.0E-7)
ax.set_ylim(-1.5E-7, 1.5E-7)

ax.set_xlabel(r'$t/T$')
ax.set_ylabel(r'$p\prime$')
ax.set_title(rf"$M_\infty={M0:.1f}$", loc='right')
ax.legend(loc='upper right', ncol=3, frameon=False, fontsize=15)